# 🏥 Medical Spell Corrector — All Bugs Fixed & Improved


## Environment, Seeds & Logging 

In [1]:
# ═══════════════════════════════════════════════════════════
# ENVIRONMENT SNAPSHOT
# ═══════════════════════════════════════════════════════════
import sys, platform, importlib, datetime
print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Date/Time : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python    : {sys.version}")
print(f"Platform  : {platform.platform()}")
for pkg in ["torch", "transformers", "datasets", "numpy",
            "pandas", "nltk", "requests"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:<18} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  {pkg:<18} NOT INSTALLED")
print("=" * 60)

ENVIRONMENT INFO
Date/Time : 2026-04-06 01:55:43
Python    : 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Platform  : Windows-10-10.0.26100-SP0
  torch              2.7.1+cu118


C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  transformers       4.40.2
  datasets           2.19.1
  numpy              1.26.4
  pandas             2.3.3
  nltk               3.9.3
  requests           2.32.5


In [2]:
import random
import numpy as np
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"Seeds set (SEED={SEED}). CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print(f"Seeds set (SEED={SEED}). PyTorch not found.")

Seeds set (SEED=42). CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# ═══════════════════════════════════════════════════════════
# Original had no logging at all; errors were silently swallowed.
# ═══════════════════════════════════════════════════════════
import logging

LOG_FILE = "spell_corrector.log"
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)
logger.info("Pipeline started")
print(f"Logging → {LOG_FILE}")

2026-04-06 01:55:59,451 | INFO     | Pipeline started
Logging → spell_corrector.log


In [4]:
import time, json, re, string, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import requests
import torch
import nltk

from datasets import Dataset
from transformers import (
    AutoTokenizer,                        
    T5ForConditionalGeneration,
    Trainer, TrainingArguments,
    EarlyStoppingCallback,                
)
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

for resource in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource.split('/')[-1], quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
logger.info(f"Device: {device}")

W0406 01:56:08.752000 18808 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
2026-04-06 01:56:11,696 | INFO     | Device: cuda


## ── SECTION 1: Drug Fetching ──

In [5]:
FALLBACK_DRUGS = [
    # Analgesics / Antipyretics
    "paracetamol", "ibuprofen", "aspirin", "naproxen", "diclofenac",
    "tramadol", "codeine", "morphine", "oxycodone", "ketorolac",
    # Antibiotics
    "amoxicillin", "azithromycin", "ciprofloxacin", "doxycycline",
    "metronidazole", "ceftriaxone", "clindamycin", "vancomycin",
    "trimethoprim", "nitrofurantoin", "penicillin", "erythromycin",
    # Cardiovascular
    "lisinopril", "amlodipine", "atenolol", "metoprolol", "losartan",
    "valsartan", "ramipril", "bisoprolol", "carvedilol", "diltiazem",
    "furosemide", "spironolactone", "hydrochlorothiazide", "digoxin",
    "warfarin", "rivaroxaban", "apixaban", "clopidogrel", "atorvastatin",
    "simvastatin", "rosuvastatin", "pravastatin",
    # Diabetes
    "metformin", "insulin", "glipizide", "glyburide", "sitagliptin",
    "empagliflozin", "dapagliflozin", "liraglutide", "pioglitazone",
    # Respiratory
    "albuterol", "salbutamol", "fluticasone", "budesonide", "salmeterol",
    "formoterol", "tiotropium", "montelukast", "cetirizine",
    "loratadine", "fexofenadine", "ipratropium",
    # GI
    "omeprazole", "pantoprazole", "lansoprazole", "ranitidine",
    "ondansetron", "metoclopramide", "loperamide", "lactulose",
    # CNS / Psychiatric
    "sertraline", "fluoxetine", "paroxetine", "citalopram", "escitalopram",
    "amitriptyline", "venlafaxine", "duloxetine", "quetiapine",
    "olanzapine", "risperidone", "haloperidol", "diazepam",
    "lorazepam", "alprazolam", "zolpidem", "gabapentin", "pregabalin",
    # Hormones / Other
    "prednisone", "prednisolone", "dexamethasone", "hydrocortisone",
    "levothyroxine", "methotrexate", "hydroxychloroquine", "allopurinol",
    "colchicine", "tamoxifen",
]

def fetch_drugs() -> list:
    """Fetch drug names from RxNorm API; merge with fallback list."""
    api_drugs = set()
    logger.info("Fetching drugs from RxNorm API ...")
    print("Fetching real drugs from RxNorm API...")
    for letter in "abcdefghijklmnopqrstuvwxyz":
        try:
            url = f"https://rxnav.nlm.nih.gov/REST/drugs.json?name={letter}"
            res = requests.get(url, timeout=5).json()
            for group in res.get("drugGroup", {}).get("conceptGroup", []):
                for item in group.get("conceptProperties", []):
                    api_drugs.add(item["name"].lower().strip())
            time.sleep(0.1)
        except Exception:
            continue

    # FIX: merge API results with fallback instead of replacing
    all_drugs = list(set(FALLBACK_DRUGS) | api_drugs)
    logger.info(f"API drugs: {len(api_drugs)}, merged total: {len(all_drugs)}")
    print(f"API fetched: {len(api_drugs)} | Fallback: {len(FALLBACK_DRUGS)} | "
          f"Merged: {len(all_drugs)}")
    return all_drugs

drugs = fetch_drugs()

2026-04-06 01:56:18,198 | INFO     | Fetching drugs from RxNorm API ...
Fetching real drugs from RxNorm API...
2026-04-06 01:57:06,967 | INFO     | API drugs: 94, merged total: 195
API fetched: 94 | Fallback: 101 | Merged: 195


## Dataset Generation ──



In [6]:
# Keyboard proximity map for realistic substitution errors
KEYBOARD_NEIGHBORS = {
    'a': 'sqwz', 'b': 'vghn', 'c': 'xdfv', 'd': 'srfce',
    'e': 'wrsdf', 'f': 'drtgv', 'g': 'ftyhb', 'h': 'gyujn',
    'i': 'uojk', 'j': 'huikm', 'k': 'jiol', 'l': 'kop',
    'm': 'njk', 'n': 'bhjm', 'o': 'iklp', 'p': 'ol',
    'q': 'wa', 'r': 'edft', 's': 'awedxz', 't': 'rfgy',
    'u': 'yhji', 'v': 'cfgb', 'w': 'qase', 'x': 'zsdc',
    'y': 'tghu', 'z': 'asx',
}

def swap_chars(word: str) -> str:
    """Swap two adjacent characters."""
    if len(word) < 3:
        return word
    i = random.randint(0, len(word) - 2)
    return word[:i] + word[i+1] + word[i] + word[i+2:]

def delete_char(word: str) -> str:
    """Delete a random character (simulates missing keystroke)."""
    if len(word) < 3:
        return word
    i = random.randint(0, len(word) - 1)
    return word[:i] + word[i+1:]

def insert_char(word: str) -> str:
    """Insert a duplicate character (simulates double keystroke)."""
    if len(word) < 2:
        return word
    i = random.randint(0, len(word) - 1)
    return word[:i] + word[i] + word[i:]  # double a char

def keyboard_sub(word: str) -> str:
    """Substitute a character with a keyboard-adjacent one."""
    if len(word) < 3:
        return word
    i = random.randint(0, len(word) - 1)
    c = word[i]
    neighbors = KEYBOARD_NEIGHBORS.get(c, "")
    if not neighbors:
        return word
    return word[:i] + random.choice(neighbors) + word[i+1:]

NOISE_OPS = [swap_chars, delete_char, insert_char, keyboard_sub]

# Abbreviation maps
ABBR = {
    "patient": "pt", "history": "hx", "and": "n", "with": "w/",
    "prescribed": "rx", "diagnosis": "dx", "treatment": "tx",
    "symptoms": "sx", "medication": "med", "milligrams": "mg",
    "once": "od", "twice": "bd", "three": "tds", "four": "qds",
}
DRUG_SHORT = {
    "paracetamol": "pcm", "ibuprofen": "ibu", "amoxicillin": "amox",
    "metformin": "met", "prednisone": "pred", "atorvastatin": "atorva",
    "metoprolol": "metop", "omeprazole": "omep", "azithromycin": "azith",
}

def add_noise(text: str, noise_rate: float = 0.25) -> str:
    """Apply realistic noisy transformations to a medical text string."""
    words = text.split()
    result = []
    for w in words:
        # Apply drug abbreviation
        if w in DRUG_SHORT and random.random() < 0.5:
            result.append(DRUG_SHORT[w])
            continue
        # Apply word abbreviation
        if w in ABBR and random.random() < 0.45:
            result.append(ABBR[w])
            continue
        # Apply character-level noise
        if random.random() < noise_rate and len(w) >= 3:
            op = random.choice(NOISE_OPS)
            w = op(w)
        result.append(w)
    return " ".join(result)

print("Noise functions defined.")
# Quick sanity check
test_phrase = "patient taking paracetamol for fever"
print(f"  Original : {test_phrase}")
print(f"  Noisy    : {add_noise(test_phrase)}")
print(f"  Noisy    : {add_noise(test_phrase)}")

Noise functions defined.
  Original : patient taking paracetamol for fever
  Noisy    : paatient takign pcm for fver
  Noisy    : patiet taking pparacetamol for feger


In [7]:
DISEASES = [
    "fever", "diabetes", "asthma", "hypertension", "infection",
    "pain", "cough", "anxiety", "depression", "allergies",
    "pneumonia", "bronchitis", "arthritis", "migraine", "epilepsy",
    "hypothyroidism", "hyperthyroidism", "gout", "anaemia", "eczema",
    "psoriasis", "urinary tract infection", "gastroenteritis",
    "atrial fibrillation", "heart failure", "chronic kidney disease",
    "copd", "sepsis", "stroke", "parkinson disease",
]
DOSAGES = [
    "5mg", "10mg", "20mg", "25mg", "50mg", "75mg",
    "100mg", "150mg", "200mg", "250mg", "500mg", "850mg",
    "1g", "1.5g", "2g",
]
FREQUENCIES = [
    "once daily", "twice daily", "three times daily", "four times daily",
    "od", "bd", "tds", "qds", "prn", "every 8 hours", "every 12 hours",
    "at night", "in the morning", "with meals",
]

TEMPLATES = [
    "patient taking {drug} for {disease}",
    "patient has {disease} taking {drug}",
    "history of {disease} on {drug}",
    "patient taking {drug} {dose} {freq}",
    "patient with {disease} given {drug} {dose} {freq}",
    "{drug} {dose} prescribed for {disease}",
    "patient on {drug} {freq} for {disease} history",
    "patient prescribed {drug} {dose} for {disease}",
    "{drug} {dose} {freq} for management of {disease}",
    "patient has {disease} and was started on {drug}",
    "continue {drug} {dose} {freq} for {disease}",
    "{drug} initiated for {disease} at {dose}",
    "patient with history of {disease} taking {drug} {dose}",
    "assessment shows {disease} patient on {drug}",
    "patient presents with {disease} given {drug} {dose} {freq}",
    "diagnosis of {disease} treatment {drug} {dose}",
    "{drug} and {drug2} prescribed for {disease}",
    "patient with {disease} started on {drug} {dose} {freq}",
    "review {drug} {dose} for {disease} management",
    "patient has {disease} requiring {drug}",
    "chronic {disease} managed with {drug} {dose}",
    "acute {disease} treated with {drug} {dose} {freq}",
    "patient on {drug} for {disease} follow up required",
    "{drug} {dose} {freq} commenced for {disease}",
    "patient with known {disease} on {drug} {dose}",
    "start {drug} {dose} {freq} for {disease}",
    "patient with {disease} switched to {drug} {dose}",
    "plan to increase {drug} to {dose} for {disease}",
    "patient taking {drug} and {drug2} for {disease}",
    "background of {disease} currently on {drug} {dose} {freq}",
    "patient needs {drug} {dose} for {disease}",
    "patient with severe {disease} given {drug} {dose}",
    "known case of {disease} on long term {drug}",
    "{drug} prescribed {dose} {freq} for {disease} symptoms",
    "patient with {disease} allergy being given {drug}",
    "consider {drug} {dose} for {disease} treatment",
    "patient on {drug} {dose} and {drug2} for {disease}",
    "patient admitted for {disease} started on {drug}",
    "discharge on {drug} {dose} {freq} for {disease}",
    "patient with {disease} maintained on {drug} {dose}",
    "{drug} {dose} given for {disease} episode",
    "patient with recurring {disease} prescribed {drug}",
    "patient with new onset {disease} commenced on {drug}",
    "patient tolerating {drug} {dose} for {disease} well",
    "patient requires {drug} {dose} {freq} for {disease}",
    "specialist recommended {drug} for {disease}",
    "patient uncontrolled {disease} add {drug} {dose}",
    "{drug} {dose} three times daily for {disease}",
    "patient with {disease} dose adjusted to {drug} {dose}",
    "patient on {drug} for {disease} reviewed today",
]

print(f"Templates: {len(TEMPLATES)}")
print(f"Drugs    : {len(drugs)}")
print(f"Diseases : {len(DISEASES)}")
unique_base = len(TEMPLATES) * len(drugs) * len(DISEASES)
print(f"Theoretical unique combos: ~{unique_base:,}")

Templates: 50
Drugs    : 195
Diseases : 30
Theoretical unique combos: ~292,500


In [8]:

def generate_dataset(n: int, seed: int) -> pd.DataFrame:
    """Generate n (input, output) pairs using a given random seed."""
    rng = random.Random(seed)  # isolated RNG per call
    data = []
    for _ in range(int(n * 1.3)):  # over-generate to compensate dedup
        tmpl   = rng.choice(TEMPLATES)
        drug   = rng.choice(drugs)
        drug2  = rng.choice([d for d in drugs if d != drug])
        disease= rng.choice(DISEASES)
        dose   = rng.choice(DOSAGES)
        freq   = rng.choice(FREQUENCIES)

        try:
            clean = tmpl.format(
                drug=drug, drug2=drug2, disease=disease,
                dose=dose, freq=freq
            ).lower()
        except KeyError:
            continue

        noisy = add_noise(clean)
        if noisy != clean:  # only keep samples that actually have noise
            data.append({"input": noisy, "output": clean})

    df = pd.DataFrame(data).drop_duplicates(subset=["input"])
    if len(df) > n:
        df = df.sample(n=n, random_state=seed)
    return df.reset_index(drop=True)

print("Generating TRAIN set  (seed=42)  ...")
df_train_raw = generate_dataset(40000, seed=SEED)          # train
print(f"  Train size: {len(df_train_raw):,}")

print("Generating VALIDATION set (seed=100) ...")
df_val_raw   = generate_dataset(5000,  seed=SEED + 100)    # different seed
print(f"  Val size:   {len(df_val_raw):,}")

print("Generating TEST set (seed=200) ...")
df_test_raw  = generate_dataset(5000,  seed=SEED + 200)    # different seed
print(f"  Test size:  {len(df_test_raw):,}")

# Save raw splits for later evaluation
df_train_raw.to_csv("medical_train.csv", index=False)
df_val_raw.to_csv("medical_val.csv",     index=False)
df_test_raw.to_csv("medical_test.csv",   index=False)
logger.info(f"Data split: train={len(df_train_raw)}, val={len(df_val_raw)}, test={len(df_test_raw)}")

print(f"\nSample pair:")
row = df_train_raw.iloc[0]
print(f"  Noisy  : {row['input']}")
print(f"  Clean  : {row['output']}")

Generating TRAIN set  (seed=42)  ...
  Train size: 40,000
Generating VALIDATION set (seed=100) ...
  Val size:   5,000
Generating TEST set (seed=200) ...
  Test size:  5,000
2026-04-06 01:57:28,924 | INFO     | Data split: train=40000, val=5000, test=5000

Sample pair:
  Noisy  : pt uncontrollled axiety add diclofeac 200mg
  Clean  : patient uncontrolled anxiety add diclofenac 200mg


## ── SECTION 3: Tokenisation & Model Loading ──

In [9]:
MODEL_NAME  = "t5-small"
MAX_LEN     = 64
PREFIX      = "fix: "
OUTPUT_DIR  = "./medical_spellcheck_model"

# FIX 🟢: AutoTokenizer replaces deprecated T5Tokenizer
logger.info(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ─────────────────────────────────────────────────────────
# FIX 🟠: Keep original text columns during preprocessing
# Original: preprocess() only added input_ids/labels,
# so 'input'/'output' columns disappeared after .map().
# evaluate_model() then crashed trying to read sample['input'].
# ─────────────────────────────────────────────────────────
def preprocess(example):
    inp = PREFIX + example["input"]
    out = example["output"]

    model_inputs = tokenizer(
        inp, max_length=MAX_LEN, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        text_target=out, max_length=MAX_LEN, truncation=True, padding="max_length"
    )
    # Replace pad token id in labels with -100 (ignored by loss)
    model_inputs["labels"] = [
        (tok if tok != tokenizer.pad_token_id else -100)
        for tok in labels["input_ids"]
    ]
    # FIX: keep original text so evaluation can read it
    model_inputs["src_text"] = example["input"]
    model_inputs["tgt_text"] = example["output"]
    return model_inputs

print("Preprocessing datasets ...")
ds_train = Dataset.from_pandas(df_train_raw).map(preprocess, remove_columns=["input", "output"])
ds_val   = Dataset.from_pandas(df_val_raw  ).map(preprocess, remove_columns=["input", "output"])
ds_test  = Dataset.from_pandas(df_test_raw ).map(preprocess, remove_columns=["input", "output"])

# Set format for PyTorch (keep text columns as plain Python strings)
TORCH_COLS = ["input_ids", "attention_mask", "labels"]
ds_train.set_format("torch", columns=TORCH_COLS, output_all_columns=True)
ds_val.set_format(  "torch", columns=TORCH_COLS, output_all_columns=True)
ds_test.set_format( "torch", columns=TORCH_COLS, output_all_columns=True)

print(f"Train: {len(ds_train):,}  Val: {len(ds_val):,}  Test: {len(ds_test):,}")
logger.info("Tokenisation complete")

2026-04-06 01:57:36,492 | INFO     | Loading tokenizer and model: t5-small
Model loaded. Parameters: 60,506,624
Preprocessing datasets ...


Map: 100%|█████████████████████████████████████████████████████████████████| 5000/5000 [00:07<00:00, 650.01 examples/s]

Train: 40,000  Val: 5,000  Test: 5,000
2026-04-06 01:58:56,912 | INFO     | Tokenisation complete


## ── SECTION 4: Baseline Models ──

In [10]:


# Reverse abbreviation map (abbr → full)
EXPAND_ABBR = {v: k for k, v in ABBR.items()}
EXPAND_ABBR.update({v: k for k, v in DRUG_SHORT.items()})

def baseline_identity(text: str) -> str:
    """Return input unchanged — measures how much noise exists."""
    return text

def baseline_rule(text: str) -> str:
    """Expand known abbreviations; no typo correction."""
    words  = text.split()
    result = [EXPAND_ABBR.get(w, w) for w in words]
    return " ".join(result)


# ─────────────────────────────────────────────────────────
# FIX 🟠: RICH EVALUATION METRICS
# Original: only BLEU on 100 samples.
# Fixed: char-accuracy, word-accuracy, exact-match, BLEU on full set.
# ─────────────────────────────────────────────────────────
def compute_metrics(predictions: list, references: list) -> dict:
    """Compute multiple text-correction metrics."""
    assert len(predictions) == len(references)

    exact_match   = []
    word_acc_list = []
    char_acc_list = []
    bleu_list     = []
    smooth        = SmoothingFunction().method1

    for pred, ref in zip(predictions, references):
        pred_s, ref_s = str(pred).strip().lower(), str(ref).strip().lower()

        # Exact match
        exact_match.append(1.0 if pred_s == ref_s else 0.0)

        # Word-level accuracy
        pred_words = pred_s.split()
        ref_words  = ref_s.split()
        if ref_words:
            n = min(len(pred_words), len(ref_words))
            matches = sum(p == r for p, r in zip(pred_words[:n], ref_words[:n]))
            word_acc_list.append(matches / len(ref_words))
        else:
            word_acc_list.append(1.0 if not pred_words else 0.0)

        # Character-level accuracy
        if ref_s:
            n = min(len(pred_s), len(ref_s))
            matches = sum(p == r for p, r in zip(pred_s[:n], ref_s[:n]))
            char_acc_list.append(matches / len(ref_s))
        else:
            char_acc_list.append(1.0 if not pred_s else 0.0)

        # Sentence BLEU
        try:
            bleu = sentence_bleu([ref_words], pred_words, smoothing_function=smooth)
        except Exception:
            bleu = 0.0
        bleu_list.append(bleu)

    # Corpus BLEU
    try:
        corpus_b = corpus_bleu(
            [[r.split()] for r in references],
            [p.split()   for p in predictions]
        )
    except Exception:
        corpus_b = 0.0

    return {
        "exact_match":   round(sum(exact_match)   / len(exact_match),   4),
        "word_accuracy": round(sum(word_acc_list)  / len(word_acc_list), 4),
        "char_accuracy": round(sum(char_acc_list)  / len(char_acc_list), 4),
        "bleu_sentence": round(sum(bleu_list)      / len(bleu_list),     4),
        "bleu_corpus":   round(corpus_b,                                  4),
        "n_samples":     len(predictions),
    }


# Evaluate baselines on test set (raw DataFrame)
test_inputs  = df_test_raw["input"].tolist()
test_targets = df_test_raw["output"].tolist()

EVAL_N = len(test_inputs)   # use full test set, not 100 samples

print(f"\nEvaluating baselines on {EVAL_N:,} test samples...")
logger.info(f"Baseline evaluation on {EVAL_N} samples")

id_preds   = [baseline_identity(t)     for t in test_inputs[:EVAL_N]]
rule_preds = [baseline_rule(t)         for t in test_inputs[:EVAL_N]]
refs       = test_targets[:EVAL_N]

id_metrics   = compute_metrics(id_preds,   refs)
rule_metrics = compute_metrics(rule_preds, refs)

print("\n" + "=" * 60)
print("BASELINE RESULTS")
print("=" * 60)
print(f"{'Metric':<20} {'Identity':>12} {'Rule-based':>12}")
print("-" * 45)
for k in ["exact_match", "word_accuracy", "char_accuracy", "bleu_sentence", "bleu_corpus"]:
    print(f"  {k:<18} {id_metrics[k]:>12.4f} {rule_metrics[k]:>12.4f}")
print("=" * 60)
print("T5 model must beat BOTH baselines to justify training cost.")
logger.info(f"Baseline identity: {id_metrics}")
logger.info(f"Baseline rule:     {rule_metrics}")


Evaluating baselines on 5,000 test samples...
2026-04-06 01:59:01,624 | INFO     | Baseline evaluation on 5000 samples

BASELINE RESULTS
Metric                   Identity   Rule-based
---------------------------------------------
  exact_match              0.0000       0.0604
  word_accuracy            0.7477       0.7859
  char_accuracy            0.3211       0.5048
  bleu_sentence            0.4790       0.5153
  bleu_corpus              0.5961       0.6016
T5 model must beat BOTH baselines to justify training cost.
2026-04-06 01:59:09,879 | INFO     | Baseline identity: {'exact_match': 0.0, 'word_accuracy': 0.7477, 'char_accuracy': 0.3211, 'bleu_sentence': 0.479, 'bleu_corpus': 0.5961, 'n_samples': 5000}
2026-04-06 01:59:09,883 | INFO     | Baseline rule:     {'exact_match': 0.0604, 'word_accuracy': 0.7859, 'char_accuracy': 0.5048, 'bleu_sentence': 0.5153, 'bleu_corpus': 0.6016, 'n_samples': 5000}


## Training ──



In [11]:
training_args = TrainingArguments(
    output_dir               = OUTPUT_DIR,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    num_train_epochs         = 5,              # more epochs; early stopping handles real cutoff
    evaluation_strategy      = "epoch",
    save_strategy            = "epoch",
    logging_steps            = 200,
    fp16                     = torch.cuda.is_available(),
    report_to                = "none",
    # FIX 🟡: load best model at end + early stopping
    load_best_model_at_end   = True,
    metric_for_best_model    = "eval_loss",
    greater_is_better        = False,
    save_total_limit         = 2,
    warmup_steps             = 500,
    weight_decay             = 0.01,
    seed                     = SEED,         
)

trainer = Trainer(
    model          = model,
    args           = training_args,
    train_dataset  = ds_train,
    eval_dataset   = ds_val,
    callbacks      = [EarlyStoppingCallback(early_stopping_patience=2)],  
)

logger.info("Starting training ...")
print("Starting training...")
try:
    train_result = trainer.train()
    logger.info(f"Training complete. Loss: {train_result.training_loss:.4f}")
    print(f"\nTraining complete. Final loss: {train_result.training_loss:.4f}")
except Exception as e:
    logger.error(f"Training failed: {e}")
    raise

2026-04-06 01:59:17,856 | INFO     | Starting training ...
Starting training...


Epoch,Training Loss,Validation Loss
1,0.152200,0.100187
2,0.086500,0.052914
3,0.066300,0.037859
4,0.055100,0.033091
5,0.049800,0.030678


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


2026-04-06 02:50:21,559 | INFO     | Training complete. Loss: 0.1299

Training complete. Final loss: 0.1299


## Full Evaluation ──

In [13]:

def generate_prediction(text: str,
                         num_beams: int = 4,
                         no_repeat_ngram: int = 2) -> str:
    """Run T5 inference with beam search (FIX 🟢: greedy → beam)."""
    inputs = tokenizer(
        PREFIX + text,
        return_tensors="pt",
        max_length=MAX_LEN,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LEN,
            num_beams=num_beams,                    
            no_repeat_ngram_size=no_repeat_ngram,   # prevents repetition
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Evaluate on full test set
print(f"Evaluating T5 on {EVAL_N:,} test samples ")
logger.info(f"Starting T5 evaluation on {EVAL_N} samples")

model.eval()
t5_preds = []
BATCH_LOG = max(1, EVAL_N // 10)  # log progress 10 times

for i, src in enumerate(test_inputs[:EVAL_N]):
    try:
        pred = generate_prediction(src)
    except Exception as e:
        logger.warning(f"Prediction failed at sample {i}: {e}")
        pred = src  # fall back to identity
    t5_preds.append(pred)
    if (i + 1) % BATCH_LOG == 0:
        print(f"  {i+1:>5} / {EVAL_N} done")

t5_metrics = compute_metrics(t5_preds, refs)
logger.info(f"T5 metrics: {t5_metrics}")

# ─────────────────────────────────────────────────────────
# Print comparison table
print("\n" + "=" * 70)
print("FULL EVALUATION COMPARISON")
print("=" * 70)
print(f"{'Metric':<22} {'Identity':>10} {'Rule-based':>12} {'T5 Model':>12}")
print("-" * 58)
for k in ["exact_match", "word_accuracy", "char_accuracy", "bleu_sentence", "bleu_corpus"]:
    id_v   = id_metrics[k]
    rule_v = rule_metrics[k]
    t5_v   = t5_metrics[k]
    # Mark best with ★
    best = max(id_v, rule_v, t5_v)
    markers = ["★" if v == best else " " for v in [id_v, rule_v, t5_v]]
    print(f"  {k:<20} {id_v:>9.4f}{markers[0]} {rule_v:>10.4f}{markers[1]} {t5_v:>10.4f}{markers[2]}")
print("=" * 70)
print("★ = best for that metric")

Evaluating T5 on 5,000 test samples 
2026-04-06 03:28:29,691 | INFO     | Starting T5 evaluation on 5000 samples
    500 / 5000 done
   1000 / 5000 done
   1500 / 5000 done
   2000 / 5000 done
   2500 / 5000 done
   3000 / 5000 done
   3500 / 5000 done
   4000 / 5000 done
   4500 / 5000 done
   5000 / 5000 done
2026-04-06 05:21:34,554 | INFO     | T5 metrics: {'exact_match': 0.5046, 'word_accuracy': 0.6297, 'char_accuracy': 0.6195, 'bleu_sentence': 0.5708, 'bleu_corpus': 0.0616, 'n_samples': 5000}

FULL EVALUATION COMPARISON
Metric                   Identity   Rule-based     T5 Model
----------------------------------------------------------
  exact_match             0.0000      0.0604      0.5046★
  word_accuracy           0.7477      0.7859★     0.6297 
  char_accuracy           0.3211      0.5048      0.6195★
  bleu_sentence           0.4790      0.5153      0.5708★
  bleu_corpus             0.5961      0.6016★     0.0616 
★ = best for that metric


In [14]:
# ─────────────────────────────────────────────────────────
# Per-category breakdown (abbreviation-only vs typo-only vs mixed)
# ─────────────────────────────────────────────────────────
import re as _re

def classify_sample(src: str, tgt: str) -> str:
    """Rough heuristic to classify noise type."""
    abbr_tokens = set(ABBR.values()) | set(DRUG_SHORT.values())
    has_abbr = any(w in abbr_tokens for w in src.split())
    has_typo = (src != tgt) and not has_abbr
    if has_abbr and has_typo:
        return "mixed"
    elif has_abbr:
        return "abbreviation"
    elif has_typo:
        return "typo"
    else:
        return "clean"

# Build category lists
cats = {"abbreviation": [], "typo": [], "mixed": [], "clean": []}
for src, tgt, pred in zip(test_inputs[:EVAL_N], refs[:EVAL_N], t5_preds[:EVAL_N]):
    c = classify_sample(src, tgt)
    cats[c].append((src, tgt, pred))

print("\nPer-category breakdown (T5 model):")
print(f"{'Category':<18} {'Count':>6} {'ExactMatch':>12} {'WordAcc':>10}")
print("-" * 48)
for cat, items in cats.items():
    if not items:
        continue
    preds_c = [p for _, _, p in items]
    refs_c  = [r for _, r, _ in items]
    m = compute_metrics(preds_c, refs_c)
    print(f"  {cat:<16} {len(items):>6} {m['exact_match']:>12.4f} {m['word_accuracy']:>10.4f}")
print("-" * 48)
logger.info("Per-category evaluation complete")


Per-category breakdown (T5 model):
Category            Count   ExactMatch    WordAcc
------------------------------------------------
  abbreviation       3113       0.5381     0.6676
  typo               1887       0.4494     0.5671
------------------------------------------------
2026-04-06 08:26:05,632 | INFO     | Per-category evaluation complete


## Error Analysis & Interpretability ──

In [15]:
# ─────────────────────────────────────────────────────────
# ERROR ANALYSIS — what the model gets wrong
# ─────────────────────────────────────────────────────────
errors = [
    (src, tgt, pred)
    for src, tgt, pred in zip(test_inputs[:EVAL_N], refs[:EVAL_N], t5_preds[:EVAL_N])
    if pred.strip().lower() != tgt.strip().lower()
]

print(f"Error samples: {len(errors)} / {EVAL_N} ({len(errors)/EVAL_N:.1%})")
print()

print("Top 10 errors (worst word accuracy):")
def word_acc(pred, ref):
    pw, rw = pred.split(), ref.split()
    if not rw: return 1.0
    n = min(len(pw), len(rw))
    return sum(p==r for p,r in zip(pw[:n], rw[:n])) / len(rw)

errors_scored = sorted(errors, key=lambda x: word_acc(x[2], x[1]))
print(f"{'Input (noisy)':<45} {'Expected':<35} {'Predicted':<35}")
print("-" * 115)
for src, tgt, pred in errors_scored[:10]:
    print(f"  {src[:43]:<43}  {tgt[:33]:<33}  {pred[:33]:<33}")

# Word-level error frequency
from collections import Counter
wrong_words = []
for _, tgt, pred in errors[:500]:
    for tw, pw in zip(tgt.split(), pred.split()):
        if tw != pw:
            wrong_words.append(tw)

print(f"\nTop 15 target words the model gets wrong:")
for word, cnt in Counter(wrong_words).most_common(15):
    print(f"  '{word}' missed {cnt} times")

Error samples: 2477 / 5000 (49.5%)

Top 10 errors (worst word accuracy):
Input (noisy)                                 Expected                            Predicted                          
-------------------------------------------------------------------------------------------------------------------
  alanine 8.8 mg/ml / arginine 4.89 mg/ml / g  alanine 8.8 mg/ml / arginine 4.89  alanine 8.8, arginine 4.89 mg/ml 
  alaniine 8.8 mg/mml / arginine 4.89 mg/ml /  alanine 8.8 mg/ml / arginine 4.89  alanine 8.8, arginine 4.89 mg/ml 
  alanine 8.8 mg/ml / arginine 4.89 mg/mk / c  alanine 8.8 mg/ml / arginine 4.89  alanine 8.8, arginine 4.89 mg/ml 
  alanine 8.8 mg//ml / arginnine 4.89 mg/ml /  alanine 8.8 mg/ml / arginine 4.89  alanine 8.8, arginine 4.89 mg/ml 
  aalanine 10.4 mg/mp / arginine 5.75 mg/ml /  alanine 10.4 mg/ml / arginine 5.7  alanine 10.4, arginine 5.75 mg/ml
  alanine 88.8 mg/ml / arginine 4.89 mg/ml /   alanine 8.8 mg/ml / arginine 4.89  alanine 8.8, arginine 4.89 mg/m

In [16]:
# ─────────────────────────────────────────────────────────
# COMPREHENSIVE INFERENCE TEST SUITE
# ─────────────────────────────────────────────────────────
print("\n=== Comprehensive Test Suite ===")

test_cases = {
    "1. Pure Abbreviations": [
        ("pt taking pcm 500mg bd",             "patient taking paracetamol 500mg twice daily"),
        ("hx of asthma on pred 10mg od",        "history of asthma on prednisone 10mg once daily"),
        ("pt w/ cough rx azithromycin",          "patient with cough prescribed azithromycin"),
    ],
    "2. Pure Typos": [
        ("patient takng ibuprofen for fevr",     "patient taking ibuprofen for fever"),
        ("patient has diabtes taking metformin", "patient has diabetes taking metformin"),
        ("history of infction on amoxicillin",   "history of infection on amoxicillin"),
    ],
    "3. Mixed (Typos + Abbreviations)": [
        ("pt w/ hyprtension given amox 500mg tds", "patient with hypertension given amoxicillin 500mg three times daily"),
        ("hx of pain taking tramadol",              "history of pain taking tramadol"),
        ("pt prescribed lisinopril 10mg od for hyprtension", "patient prescribed lisinopril 10mg once daily for hypertension"),
    ],
    "4. Clean Text (over-correction check)": [
        ("patient taking atorvastatin 20mg once daily", "patient taking atorvastatin 20mg once daily"),
        ("history of diabetes on insulin",              "history of diabetes on insulin"),
    ],
}

all_preds, all_refs = [], []
for category, samples in test_cases.items():
    print(f"\n--- {category} ---")
    for src, expected in samples:
        pred = generate_prediction(src)
        em   = "✓" if pred.strip().lower() == expected.strip().lower() else "✗"
        print(f"  Input    : {src}")
        print(f"  Expected : {expected}")
        print(f"  T5 Output: {pred}  {em}")
        all_preds.append(pred)
        all_refs.append(expected)
        print()
    print("=" * 55)

suite_m = compute_metrics(all_preds, all_refs)
print(f"\nTest suite summary: exact={suite_m['exact_match']:.2f}  "
      f"word_acc={suite_m['word_accuracy']:.2f}  BLEU={suite_m['bleu_sentence']:.2f}")
logger.info(f"Test suite: {suite_m}")


=== Comprehensive Test Suite ===

--- 1. Pure Abbreviations ---
  Input    : pt taking pcm 500mg bd
  Expected : patient taking paracetamol 500mg twice daily
  T5 Output: patient taking paracetamol 500mg bd  ✗

  Input    : hx of asthma on pred 10mg od
  Expected : history of asthma on prednisone 10mg once daily
  T5 Output: history of asthma on prednisone 10mg od  ✗

  Input    : pt w/ cough rx azithromycin
  Expected : patient with cough prescribed azithromycin
  T5 Output: patient with cough prescribed azithromycin  ✓


--- 2. Pure Typos ---
  Input    : patient takng ibuprofen for fevr
  Expected : patient taking ibuprofen for fever
  T5 Output: patient taking ibuprofen for fever  ✓

  Input    : patient has diabtes taking metformin
  Expected : patient has diabetes taking metformin
  T5 Output: patient has diabetes taking metformin  ✓

  Input    : history of infction on amoxicillin
  Expected : history of infection on amoxicillin
  T5 Output: history of infection on amoxicillin 

##  Model + Metadata ──

In [17]:

import datetime, json, sys

SAVE_DIR = "./medical_spellcheck_final"

try:
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

    metadata = {
        "model_name":       MODEL_NAME,
        "saved_at":         datetime.datetime.now().isoformat(),
        "seed":             SEED,
        "python_version":   sys.version.split()[0],
        "max_length":       MAX_LEN,
        "prefix":           PREFIX,
        "beam_size":        4,
        "dataset": {
            "train_size":   len(df_train_raw),
            "val_size":     len(df_val_raw),
            "test_size":    len(df_test_raw),
            "n_drugs":      len(drugs),
            "n_diseases":   len(DISEASES),
            "n_templates":  len(TEMPLATES),
        },
        "test_metrics": {
            "t5":           t5_metrics,
            "baseline_rule": rule_metrics,
            "baseline_identity": id_metrics,
        },
        "bug_fixes_applied": [
            "data-leakage-fixed-separate-seeds",
            "seed-controls-added",
            "autotokenizer-replaces-t5tokenizer",
            "early-stopping-added",
            "beam-search-replaces-greedy",
            "full-test-evaluation",
            "baselines-added",
            "richer-noise-functions",
            "original-columns-preserved",
            "rxnorm-threshold-fixed",
        ],
    }

    with open(f"{SAVE_DIR}/model_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    logger.info(f"Model saved → {SAVE_DIR}")
    print(f"✅ Model     → {SAVE_DIR}")
    print(f"✅ Metadata  → {SAVE_DIR}/model_metadata.json")
    print(json.dumps(metadata["test_metrics"], indent=2))

except Exception as e:
    logger.error(f"Save failed: {e}")
    raise

2026-04-06 08:27:38,780 | INFO     | Model saved → ./medical_spellcheck_final
✅ Model     → ./medical_spellcheck_final
✅ Metadata  → ./medical_spellcheck_final/model_metadata.json
{
  "t5": {
    "exact_match": 0.5046,
    "word_accuracy": 0.6297,
    "char_accuracy": 0.6195,
    "bleu_sentence": 0.5708,
    "bleu_corpus": 0.0616,
    "n_samples": 5000
  },
  "baseline_rule": {
    "exact_match": 0.0604,
    "word_accuracy": 0.7859,
    "char_accuracy": 0.5048,
    "bleu_sentence": 0.5153,
    "bleu_corpus": 0.6016,
    "n_samples": 5000
  },
  "baseline_identity": {
    "exact_match": 0.0,
    "word_accuracy": 0.7477,
    "char_accuracy": 0.3211,
    "bleu_sentence": 0.479,
    "bleu_corpus": 0.5961,
    "n_samples": 5000
  }
}


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 49: character maps to <undefined>
Call stack:
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_a

## Load & Verify ──

In [18]:
# ─────────────────────────────────────────────────────────
# LOAD FROM DISK AND VERIFY
# ─────────────────────────────────────────────────────────
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch, json

device       = "cuda" if torch.cuda.is_available() else "cpu"
LOAD_DIR     = "./medical_spellcheck_final"
PREFIX_LOAD  = "fix: "
MAX_LEN_LOAD = 64

print("Loading saved model...")
try:
    tokenizer_loaded = AutoTokenizer.from_pretrained(LOAD_DIR)
    model_loaded     = T5ForConditionalGeneration.from_pretrained(LOAD_DIR).to(device)
    model_loaded.eval()

    with open(f"{LOAD_DIR}/model_metadata.json") as f:
        meta = json.load(f)
    print(f"Model from: {meta['saved_at']}")
    print(f"T5 exact_match on test: {meta['test_metrics']['t5']['exact_match']}")

except Exception as e:
    print(f"Load failed: {e}")
    raise


def correct_text(text: str) -> str:
    """Correct medical text using the loaded model."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input must be a non-empty string.")
    inputs = tokenizer_loaded(
        PREFIX_LOAD + text,
        return_tensors="pt",
        max_length=MAX_LEN_LOAD,
        truncation=True,
    ).to(device)
    with torch.no_grad():
        out = model_loaded.generate(
            **inputs,
            max_length=MAX_LEN_LOAD,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )
    return tokenizer_loaded.decode(out[0], skip_special_tokens=True)


print("\n--- Final Inference Verification ---")
test_phrases = [
    "pt needs amox for severe infction",
    "pt takng pcm for fevr",
    "asthma patient taking prednison",
    "hx of hypertension on lsartan 50mg",
]
for phrase in test_phrases:
    result = correct_text(phrase)
    print(f"  Input : {phrase}")
    print(f"  Output: {result}")
    print()

Loading saved model...
Model from: 2026-04-06T08:27:38.780526
T5 exact_match on test: 0.5046

--- Final Inference Verification ---
  Input : pt needs amox for severe infction
  Output: patient needs amoxicillin for severe infection

  Input : pt takng pcm for fevr
  Output: patient taking paracetamol for fever

  Input : asthma patient taking prednison
  Output: asthma patient taking prednison

  Input : hx of hypertension on lsartan 50mg
  Output: history of hypertension on lisartan 50mg

